# AIDP Race Intelligence: Executive visualization

A read-only, Spark-first view of governed Gold outputs. Historical evidence and synthetic WHAT-IF scenario outputs remain separate. Virtual Max is synthetic only; timing catch is not a physical overtake model.

## Load governed Gold inputs

This notebook runs after AIDP notebooks 00–07 and reads no local files or unmanaged outputs.

In [ ]:
from pathlib import Path
import os
import sys

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')

sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import load_aidp_config, table_name

config = load_aidp_config(root / 'config/settings.yaml')

In [ ]:
gold_inputs = {
    'observed_driver_profiles': table_name(config, 'gold_historical_schema', 'observed_driver_profiles'),
    'historical_pace_summary': table_name(config, 'gold_historical_schema', 'historical_pace_summary'),
    'virtual_max_profile': table_name(config, 'gold_scenario_schema', 'virtual_max_profile'),
    'monte_carlo_scenario_summary': table_name(config, 'gold_scenario_schema', 'monte_carlo_scenario_summary'),
    'monte_carlo_probability_curve': table_name(config, 'gold_scenario_schema', 'monte_carlo_probability_curve'),
}
missing = [name for name, target in gold_inputs.items() if not spark.catalog.tableExists(target)]
if missing:
    raise RuntimeError(f'Missing required governed Gold input(s): {", ".join(missing)}')

profiles = spark.table(gold_inputs['observed_driver_profiles'])
historical_pace_summary = spark.table(gold_inputs['historical_pace_summary'])
virtual_max_profile = spark.table(gold_inputs['virtual_max_profile'])
scenario_summary = spark.table(gold_inputs['monte_carlo_scenario_summary'])
probability_curve = spark.table(gold_inputs['monte_carlo_probability_curve'])

scenario = scenario_summary.orderBy(F.col('simulations').desc()).first()
virtual_max = virtual_max_profile.first()
if scenario is None or virtual_max is None:
    raise RuntimeError('Current Gold Scenario summary or Virtual Max profile is empty.')
if probability_curve.limit(1).count() == 0:
    raise RuntimeError('Current Gold probability curve is empty.')

print('Governed Gold inputs ready.')
print('Historical pace summary rows:', historical_pace_summary.count())

## 1. Executive summary

All values below are read dynamically from the current Gold Historical and Gold Scenario tables.

In [ ]:
executive_summary = virtual_max_profile.select(
    F.col('virtual_max_pace_seconds').alias('virtual_max_pace_seconds'),
    F.col('assumed_advantage_seconds_per_lap').alias('assumed_advantage_seconds_per_lap'),
    F.col('scenario_type'),
    F.col('label'),
).crossJoin(
    scenario_summary.orderBy(F.col('simulations').desc()).limit(1).select(
        'supported_field_size',
        'race_laps',
        'deterministic_max_delay_seconds',
        'probability_at_zero_delay',
        'delay_50pct',
        'delay_80pct',
        'delay_90pct',
        'simulations',
    )
)
executive_summary.show(truncate=False)

Positive threshold: Virtual Max can start that many seconds later.

Negative threshold: Virtual Max would require that many seconds of head start.

**Timing catch only; not a physical overtake model.**

## 2. Historical field

Driver IDs are source identifiers only. No real-world driver identity is assigned in this notebook.

In [ ]:
supported_profiles = (
    profiles
    .filter(F.col('support_status') == 'sufficient')
    .select('driver_id', 'pooled_median_pace', 'support_status')
    .orderBy(F.col('pooled_median_pace').asc())
)

supported_field_size = supported_profiles.count()
fastest = supported_profiles.first()
if fastest is None:
    raise RuntimeError('No supported historical driver profile is available.')

fastest_observed_pace = float(fastest.pooled_median_pace)
virtual_max_pace = float(virtual_max.virtual_max_pace_seconds)
print('Supported historical drivers:', supported_field_size)
print('Fastest observed historical pace:', fastest_observed_pace)
print('Virtual Max synthetic pace:', virtual_max_pace)
print('Fastest observed minus Virtual Max pace:', fastest_observed_pace - virtual_max_pace)
supported_profiles.show(supported_field_size, truncate=False)

## 3. Full-field timing-catch probability

The presentation table selects integer-second delays plus the dynamic P50, P80, P90, zero-delay, and deterministic-delay reference points. It is deliberately kept as a governed Spark table for the AIDP visualization UI; no plotting package is installed or required.

In [ ]:
threshold_values = [
    float(value)
    for value in (
        scenario.delay_50pct,
        scenario.delay_80pct,
        scenario.delay_90pct,
        scenario.deterministic_max_delay_seconds,
        0.0,
    )
    if value is not None
]

important_delay = F.lit(False)
for delay in threshold_values:
    important_delay = important_delay | (F.abs(F.col('delay_seconds') - F.lit(delay)) < F.lit(1e-9))

presentation_curve = (
    probability_curve
    .filter(
        (F.abs(F.col('delay_seconds') - F.round(F.col('delay_seconds'), 0)) < F.lit(1e-9))
        | important_delay
    )
    .select('delay_seconds', 'full_field_catch_probability')
    .orderBy('delay_seconds')
)
presentation_curve.show(60, truncate=False)

Reference points are dynamic Gold Scenario values: P50, P80, P90, zero-delay probability, and deterministic maximum delay. As delay increases, the full-field timing-catch probability should not increase.

## 4. Deterministic chase

This is a **Deterministic timing-gap illustration**, using the fastest supported observed historical pace as the limiting opponent. It is not an overtake trajectory.

In [ ]:
race_laps = int(scenario.race_laps)
deterministic_delay = float(scenario.deterministic_max_delay_seconds)
pace_gain_per_lap = fastest_observed_pace - virtual_max_pace
if pace_gain_per_lap <= 0:
    raise RuntimeError('Virtual Max synthetic pace must be faster than the limiting observed pace.')

deterministic_chase = (
    spark.range(0, race_laps + 1)
    .select(
        F.col('id').cast('int').alias('lap_number'),
        (
            F.lit(deterministic_delay)
            - F.col('id') * F.lit(pace_gain_per_lap)
        ).cast('double').alias('remaining_gap_seconds'),
    )
)
final_gap = deterministic_chase.orderBy(F.col('lap_number').desc()).first().remaining_gap_seconds
print('Final deterministic timing gap:', final_gap)
deterministic_chase.show(race_laps + 1, truncate=False)

## 5. Scenario interpretation

The deterministic result represents theoretical central-pace capability against the limiting observed pace. The Monte Carlo curve represents robustness under centered historical within-driver variability applied to the observed field. Virtual Max remains deterministic because no observed Virtual Max variability exists.

P50 can permit a delayed start. Negative P80 or P90 thresholds mean a head start is required at those confidence levels. These synthetic assumptions are not historical evidence about Max Verstappen.

## 6. Final executive table

Presentation values are formatted only after the governed Gold results have been read; no Gold or Silver data is written by this notebook.

In [ ]:
final_executive_table = spark.createDataFrame([
    ('Deterministic maximum delay', f'{deterministic_delay:.3f} seconds', 'Theoretical central-pace delayed start against the limiting observed pace.'),
    ('P50 threshold', f'{float(scenario.delay_50pct):.3f} seconds' if scenario.delay_50pct is not None else 'not available', 'Maximum delay with at least 50% full-field timing-catch probability.'),
    ('P80 threshold', f'{float(scenario.delay_80pct):.3f} seconds' if scenario.delay_80pct is not None else 'not available', 'Negative value means a required head start for at least 80% probability.'),
    ('P90 threshold', f'{float(scenario.delay_90pct):.3f} seconds' if scenario.delay_90pct is not None else 'not available', 'Negative value means a required head start for at least 90% probability.'),
    ('Probability at zero delay', f'{float(scenario.probability_at_zero_delay):.3f}', 'Full-field timing-catch probability when Virtual Max starts simultaneously.'),
], 'Metric string, Value string, Interpretation string')
final_executive_table.show(truncate=False)
print('Historical evidence remains in Gold Historical; Virtual Max and timing-catch outputs remain synthetic Gold Scenario artifacts.')

In [ ]:
gold_tables = [
    table_name(
        config,
        "gold_historical_schema",
        "observed_driver_profiles"
    ),
    table_name(
        config,
        "gold_scenario_schema",
        "virtual_max_profile"
    ),
    table_name(
        config,
        "gold_scenario_schema",
        "monte_carlo_scenario_summary"
    ),
    table_name(
        config,
        "gold_scenario_schema",
        "monte_carlo_probability_curve"
    ),
]

for table in gold_tables:
    print("\n================================")
    print(table)
    print("================================")

    spark.sql(
        f"DESCRIBE DETAIL {table}"
    ).select(
        "format",
        "location"
    ).show(
        truncate=False
    )